In [1]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()

spark = SparkSession. \
builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [2]:
sc = spark.sparkContext
covid_cases_rdd = sc.textFile("data/datasets/covid19/cases/covid_dataset_cases.csv")
covid_states_rdd = sc.textFile("data/datasets/covid19/states/covid_dataset_states.csv")

### 4. The top 3 States having least no.of deaths

In [9]:
# state, deathConfirmed
covid_state_death_map = covid_cases_rdd.map(lambda x: (x.split(",")[1], int(x.split(",")[23])))

In [10]:
covid_state_death_map.take(5)

[('AP', 42), ('AP', 45), ('HP', 30), ('HP', 7), ('AS', 9)]

In [11]:
covid_state_death_agg = covid_state_death_map.reduceByKey(lambda x,y: x+y)

In [12]:
covid_state_death_agg.sortBy(lambda x: x[1], ascending=True).take(3)

[('AS', 9), ('JH', 10), ('CG', 31)]

In [13]:
spark.stop()